# LAMPOSE — train Kavya's voice (Piper)

Run the cells **one at a time, top to bottom**. Click ▶ on a cell, wait for
the green tick, then do the next one.

If a cell shows a red **Error**, stop and send the text to Claude.


## Step 1 — check we have a GPU
Menu **Runtime → Change runtime type → T4 GPU → Save**, then run this.
It must print a table containing `Tesla T4`.


In [ ]:
!nvidia-smi


## Step 2 — install Piper

About 4 minutes. Ignore yellow warnings and 'dependency conflict' lines —
the last line must say **install finished**, and the check below it must say
**piper.train is importable**.


In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1
%cd /content/piper1
!pip install -q -e '.[train]'
!bash build_monotonic_align.sh
print('\ninstall finished')


In [ ]:
# proves the install really worked — do not continue unless this passes
import importlib, subprocess
importlib.import_module('piper.train')
print('piper.train is importable')
print(subprocess.run(['espeak-ng','-v','te','-q','--ipa','నమస్తే సర్'],
                     capture_output=True, text=True).stdout.strip(),
      '  <- Telugu phonemes, espeak is working')


## Step 3 — upload the recordings

Run this, click **Choose Files**, pick `piper_dataset.zip` from your Desktop.
It is 54 MB, so allow a few minutes. Do not switch tabs while it uploads.


In [ ]:
from google.colab import files
up = files.upload()          # choose piper_dataset.zip
print(list(up))


In [ ]:
# unzip — the last line must print 408
!rm -rf /content/dataset /content/tmp && mkdir -p /content/dataset /content/tmp
!unzip -q -o piper_dataset.zip -d /content/tmp
!mv /content/tmp/piper/* /content/dataset/
!head -2 /content/dataset/metadata.csv
!ls /content/dataset/wavs | wc -l


### Only if the upload above failed: use Google Drive
Put the zip in your Drive first, then delete the `# ` from each line and run.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !rm -rf /content/dataset /content/tmp && mkdir -p /content/dataset /content/tmp
# !unzip -q -o '/content/drive/MyDrive/piper_dataset.zip' -d /content/tmp
# !mv /content/tmp/piper/* /content/dataset/
# !ls /content/dataset/wavs | wc -l


## Step 4 — download the starting voice

We start from an English voice and teach it hers. This is why 24 minutes
of audio is enough. The file is about 1 GB — a few minutes.


In [ ]:
!wget -q --show-progress -O /content/base.ckpt \
  'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!ls -lh /content/base.ckpt


### Step 4b — make the checkpoint loadable

Run this before Step 5. It must print **checkpoint cleaned and verified**.


In [ ]:
# PyTorch 2.6 refuses to load checkpoints containing anything but tensors,
# and this one stores a file path inside it. Rewrite those paths as plain
# text, then prove it loads the strict way the trainer will.
import torch, pathlib

def clean(o):
    if isinstance(o, pathlib.PurePath): return str(o)
    if isinstance(o, dict):  return {k: clean(v) for k, v in o.items()}
    if isinstance(o, list):  return [clean(v) for v in o]
    if isinstance(o, tuple): return tuple(clean(v) for v in o)
    return o

ck = torch.load('/content/base.ckpt', map_location='cpu', weights_only=False)
torch.save(clean(ck), '/content/base_clean.ckpt')
torch.load('/content/base_clean.ckpt', map_location='cpu', weights_only=True)
print('checkpoint cleaned and verified — Step 5 can run now')


## Step 5 — train

**The long one.** Let it run **1–2 hours** for this practice run.

You will see `Epoch 0:` with a progress bar counting up — that means it is
working. Keep this tab open.

**To stop:** press the ■ button on the cell. That is safe — it saves as it
goes, and Step 6 picks up whatever it has learned.


In [ ]:
%cd /content/piper1
!python3 -m piper.train fit \
  --data.voice_name kavya \
  --data.csv_path /content/dataset/metadata.csv \
  --data.audio_dir /content/dataset/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice te \
  --data.cache_dir /content/cache \
  --data.config_path /content/kavya.json \
  --data.batch_size 16 \
  --model.warmstart_ckpt /content/base_clean.ckpt


## Step 6 — make the voice file


In [ ]:
import glob, os, shutil
cks = sorted(glob.glob('/content/piper1/lightning_logs/**/*.ckpt', recursive=True)
             + glob.glob('/content/**/checkpoints/*.ckpt', recursive=True),
             key=os.path.getmtime)
print('checkpoints found:', len(cks))
assert cks, 'No checkpoint saved yet — let Step 5 run longer.'
last = cks[-1]; print('using:', last)
!python3 -m piper.train.export_onnx --checkpoint '{last}' --output-file /content/kavya.onnx
shutil.copy('/content/kavya.json', '/content/kavya.onnx.json')
!ls -lh /content/kavya.onnx*


## Step 7 — listen to her
The moment of truth. It will sound rough after a short run — that is expected.


In [ ]:
!pip -q install piper-tts
!echo 'హలో నమస్తే సర్! మీరు సుమా ఓనర్ గారేనా?' | \
  python3 -m piper -m /content/kavya.onnx -f /content/test.wav
from IPython.display import Audio, display
display(Audio('/content/test.wav'))


## Step 8 — download
Two files. Send **both** to Claude.


In [ ]:
from google.colab import files
files.download('/content/kavya.onnx')
files.download('/content/kavya.onnx.json')
